In [1]:
# ============================================================================
# EMIPredict AI - Fast Leakage-Proof Regression Pipeline with MLflow
# ============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold,cross_validate
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, PowerTransformer, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from scipy.stats import randint, uniform

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow import MlflowClient

from regression_feature_engineering import RegressionFeatureEngineer


In [17]:
# ======================================
# SETUP
# ======================================
mlflow.set_tracking_uri("http://13.204.193.251:5000")
mlflow.set_experiment("EMI_Regression_Experiment")

<Experiment: artifact_location='s3://mlflow-tracking-loan/4', creation_time=1764507718248, experiment_id='4', last_update_time=1764507718248, lifecycle_stage='active', name='EMI_Regression_Experiment', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [3]:

df = pd.read_csv(r"D:\backup\AI course work\Guvi\Assignement_2\data\clean_emi_data.csv")
df.dropna(subset=["max_monthly_emi"], inplace=True)

X = df.drop(["emi_eligibility", "max_monthly_emi"], axis=1)
y = df["max_monthly_emi"]

print(f"Data: {df.shape}")
print(f"Target Stats: Mean={y.mean():.2f}, Std={y.std():.2f}, Min={y.min():.2f}, Max={y.max():.2f}\n")

Data: (404800, 27)
Target Stats: Mean=6440.63, Std=6648.85, Min=500.00, Max=23730.00



In [4]:
# ======================================
# TRAIN-TEST SPLIT
# ======================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}\n")

Train: (323840, 25), Test: (80960, 25)



In [5]:
# ======================================
# PREPROCESSING
# ======================================
categorical_cols = ["gender", "marital_status", "education", "employment_type",
                   "company_type", "house_type", "existing_loans", "emi_scenario"]

nominal_cols = ["gender", "marital_status", "employment_type", "company_type", "house_type", "emi_scenario"]
ordinal_cols = ["education"]
binary_cols = ["existing_loans"]
education_order = ["High School", "Graduate", "Professional", "Post Graduate"]

# Determine skewness
fe_temp = RegressionFeatureEngineer()
tmp = fe_temp.fit_transform(X_train)
num_cols = tmp.select_dtypes(include=["int64", "float64"]).columns
sk = tmp[num_cols].skew()

low_skew = sk[abs(sk) <= 0.5].index.tolist()
mid_skew = sk[(abs(sk) > 0.5) & (abs(sk) <= 1)].index.tolist()
high_skew = sk[abs(sk) > 1].index.tolist()

preprocessor = ColumnTransformer([
    ("low", Pipeline([("impute", SimpleImputer(strategy="median")), 
                      ("scale", StandardScaler())]), low_skew),
    ("mid", Pipeline([("impute", SimpleImputer(strategy="median")), 
                      ("power", PowerTransformer()), 
                      ("scale", StandardScaler())]), mid_skew),
    ("high", Pipeline([("impute", SimpleImputer(strategy="median")), 
                       ("quantile", QuantileTransformer(output_distribution="normal")), 
                       ("scale", StandardScaler())]), high_skew),
    ("nominal", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))]), nominal_cols),
    ("ordinal", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("ord", OrdinalEncoder(categories=[education_order], handle_unknown='use_encoded_value', unknown_value=-1))]), ordinal_cols),
    ("binary", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                         ("ord", OrdinalEncoder(categories=[["No", "Yes"]], handle_unknown='use_encoded_value', unknown_value=-1))]), binary_cols)
], verbose_feature_names_out=False)

In [6]:
# ======================================
# BUILD PIPELINE
# ======================================
def build_pipeline(model):
    selector = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), threshold='median')
    return Pipeline([
        ("feature_eng", RegressionFeatureEngineer()),
        ("preprocess", preprocessor),
        ("select", selector),
        ("model", model)
    ])


In [26]:

# ======================================
# BASELINE EVALUATION
# ======================================
print("="*60)
print("BASELINE MODEL EVALUATION")
print("="*60)

models = {
    "Linear_Regression": LinearRegression(),
    "Random_Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    "Decision_Tree": DecisionTreeRegressor(random_state=42),
    "Gradient_Boosting": GradientBoostingRegressor(random_state=42)
}

baseline_results = {}

for name, model in models.items():
    print(f"\n{name}...")
    
    with mlflow.start_run(run_name=f"Baseline_{name}"):
        mlflow.log_param("model_type", name)
        
        pipe = build_pipeline(model)
        pipe.fit(X_train, y_train)
        
        y_test_pred = pipe.predict(X_test)
        
        rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        mae = mean_absolute_error(y_test, y_test_pred)
        r2 = r2_score(y_test, y_test_pred)
        mape = mean_absolute_percentage_error(y_test, y_test_pred) * 100
        
        mlflow.log_metric("test_rmse", rmse)
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_r2", r2)
        mlflow.log_metric("test_mape", mape)
        
        # Prediction vs Actual plot
        fig, ax = plt.subplots(figsize=(8, 6))
        plt.scatter(y_test, y_test_pred, alpha=0.5)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        plt.xlabel('Actual')
        plt.ylabel('Predicted')
        plt.title(f'{name} - Predictions vs Actual')
        mlflow.log_figure(fig, f"pred_vs_actual_{name}.png")
        plt.close()
        
        # Residuals plot
        fig, ax = plt.subplots(figsize=(8, 6))
        residuals = y_test - y_test_pred
        plt.scatter(y_test_pred, residuals, alpha=0.5)
        plt.axhline(y=0, color='r', linestyle='--', lw=2)
        plt.xlabel('Predicted')
        plt.ylabel('Residuals')
        plt.title(f'{name} - Residual Plot')
        mlflow.log_figure(fig, f"residuals_{name}.png")
        plt.close()
        
        # Log dataset info as artifact
        if name == list(models.keys())[0]:  # Only once
            dataset_info = {
                'train_shape': X_train.shape,
                'test_shape': X_test.shape,
                'target_mean': float(y_train.mean()),
                'target_std': float(y_train.std()),
                'features': X_train.columns.tolist()
            }
            with open("dataset_info_regression.txt", "w") as f:
                for key, value in dataset_info.items():
                    f.write(f"{key}: {value}\n")
            mlflow.log_artifact("dataset_info_regression.txt")
        
        baseline_results[name] = {'rmse': rmse, 'mae': mae, 'r2': r2, 'mape': mape}
        
        print(f"  RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}, MAPE: {mape:.2f}%")

results_df = pd.DataFrame(baseline_results).T.sort_values('r2', ascending=False)
print("\n" + "="*60)
print("BASELINE RESULTS")
print("="*60)
print(results_df)

BASELINE MODEL EVALUATION

Linear_Regression...
  RMSE: 3266.76, MAE: 2456.15, R2: 0.7569, MAPE: 179.71%
🏃 View run Baseline_Linear_Regression at: http://13.204.193.251:5000/#/experiments/4/runs/2589094716d64250973034d6ce92ca5b
🧪 View experiment at: http://13.204.193.251:5000/#/experiments/4

Random_Forest...
  RMSE: 1123.44, MAE: 443.07, R2: 0.9712, MAPE: 11.74%
🏃 View run Baseline_Random_Forest at: http://13.204.193.251:5000/#/experiments/4/runs/9266db563b25457b9f23c6f33a50b1e6
🧪 View experiment at: http://13.204.193.251:5000/#/experiments/4

XGBoost...
  RMSE: 993.33, MAE: 472.88, R2: 0.9775, MAPE: 15.47%
🏃 View run Baseline_XGBoost at: http://13.204.193.251:5000/#/experiments/4/runs/7f94d991fa8a4f78aa778deeee5f2c38
🧪 View experiment at: http://13.204.193.251:5000/#/experiments/4

Decision_Tree...
  RMSE: 1593.19, MAE: 582.67, R2: 0.9422, MAPE: 13.74%
🏃 View run Baseline_Decision_Tree at: http://13.204.193.251:5000/#/experiments/4/runs/0ee3a4564d974827a48948fe9ef9d4f4
🧪 View experim

In [28]:


best_model_name = results_df.index[0]
print(f"\n✓ Best Model: {best_model_name}\n")



✓ Best Model: XGBoost



In [9]:
models = {
    "Linear_Regression": LinearRegression(),
    "Random_Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    "Decision_Tree": DecisionTreeRegressor(random_state=42),
    "Gradient_Boosting": GradientBoostingRegressor(random_state=42)
}

In [10]:
best_model_name  = "XGBoost"

In [13]:
# ==========================================================
# REPLACEMENT BLOCK — NO HYPERPARAMETER TUNING FOR REGRESSION
# USING K-FOLD CROSS-VALIDATION FOR FINAL MODEL EVALUATION
# ==========================================================

print("\n" + "="*70)
print(f"FINAL REGRESSION MODEL SELECTION (NO TUNING REQUIRED) → {best_model_name}")
print("="*70)

final_model = models[best_model_name]
final_pipe = build_pipeline(final_model)

# ---------- 1. CROSS-VALIDATION ----------
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_validate(
    final_pipe,
    X_train,
    y_train,
    cv=cv,
    scoring={
        'rmse': 'neg_root_mean_squared_error',
        'mae': 'neg_mean_absolute_error',
        'r2': 'r2',
        'mape': 'neg_mean_absolute_percentage_error'
    },
    n_jobs=-1,
    return_train_score=False
)

# Convert negative scoring to positive
cv_rmse = -cv_scores['test_rmse']
cv_mae = -cv_scores['test_mae']
cv_mape = -cv_scores['test_mape'] * 100  # percentage

print("\nCROSS-VALIDATION PERFORMANCE (5-Fold):")
print(f"RMSE:     {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")
print(f"MAE:      {cv_mae.mean():.4f} ± {cv_mae.std():.4f}")
print(f"MAPE(%):  {cv_mape.mean():.4f} ± {cv_mape.std():.4f}")
print(f"R2:       {cv_scores['test_r2'].mean():.4f} ± {cv_scores['test_r2'].std():.4f}")

# ---------- 2. FIT FINAL MODEL ON ALL TRAINING DATA ----------
final_pipe.fit(X_train, y_train)

# ---------- 3. FINAL TEST EVALUATION ----------
y_test_pred = final_pipe.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)
mape = mean_absolute_percentage_error(y_test, y_test_pred) * 100

print("\nFINAL TEST PERFORMANCE:")
print(f"Test RMSE:     {rmse:.4f}")
print(f"Test MAE:      {mae:.4f}")
print(f"Test MAPE(%):  {mape:.4f}")
print(f"Test R2:       {r2:.4f}")

# ---------- 4. MLflow Logging ----------
with mlflow.start_run(run_name=f"Final_{best_model_name}_Regression_No_Tuning"):

    mlflow.log_param("model_type", best_model_name)
    mlflow.log_param("hyperparameter_tuning", "Not Required — Strong Baseline")
    mlflow.log_param("cv_type", "KFold(5)")

    # CV metrics
    mlflow.log_metric("cv_rmse_mean", cv_rmse.mean())
    mlflow.log_metric("cv_mae_mean", cv_mae.mean())
    mlflow.log_metric("cv_mape_mean", cv_mape.mean())
    mlflow.log_metric("cv_r2_mean", cv_scores['test_r2'].mean())

    # Test metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_mae", mae)
    mlflow.log_metric("test_mape", mape)
    mlflow.log_metric("test_r2", r2)

    # ---- Final Prediction vs Actual Plot ----
    fig, ax = plt.subplots(figsize=(10, 8))
    plt.scatter(y_test, y_test_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Actual EMI')
    plt.ylabel('Predicted EMI')
    plt.title(f'Final Prediction vs Actual - {best_model_name}')
    mlflow.log_figure(fig, "final_pred_vs_actual.png")
    plt.close()

    # ---- Residual Plot ----
    residuals = y_test - y_test_pred
    fig, ax = plt.subplots(figsize=(10, 8))
    plt.scatter(y_test_pred, residuals, alpha=0.5)
    plt.axhline(y=0, color='r', linestyle='--', lw=2)
    plt.xlabel('Predicted EMI')
    plt.ylabel('Residuals')
    plt.title(f'Residuals Plot - {best_model_name}')
    mlflow.log_figure(fig, "final_residuals.png")
    plt.close()

    # ---- SAVE & REGISTER MODEL ON EC2 ----
    signature = infer_signature(X_train, final_pipe.predict(X_train))

    mlflow.sklearn.log_model(
        final_pipe,
        "model",
        signature=signature,
        registered_model_name=f"EMI_Regression_{best_model_name}"
    )

print("\n🎯 FINAL REGRESSION MODEL SAVED — NO TUNING — CV VALIDATED\n")



FINAL REGRESSION MODEL SELECTION (NO TUNING REQUIRED) → XGBoost

CROSS-VALIDATION PERFORMANCE (5-Fold):
RMSE:     1001.8430 ± 9.5976
MAE:      473.7729 ± 4.8219
MAPE(%):  15.7040 ± 0.1917
R2:       0.9773 ± 0.0005

FINAL TEST PERFORMANCE:
Test RMSE:     987.7624
Test MAE:      473.5182
Test MAPE(%):  15.7528
Test R2:       0.9778


2025/12/01 13:02:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'EMI_Regression_XGBoost'.
2025/12/01 13:13:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: EMI_Regression_XGBoost, version 1
Created version '1' of model 'EMI_Regression_XGBoost'.


🏃 View run Final_XGBoost_Regression_No_Tuning at: http://13.204.193.251:5000/#/experiments/4/runs/6c2944766afc4afcb6be5ecee04085fb
🧪 View experiment at: http://13.204.193.251:5000/#/experiments/4

🎯 FINAL REGRESSION MODEL SAVED — NO TUNING — CV VALIDATED



In [18]:
# ---- Transition Model to Production ----
client = MlflowClient()

try:
    latest_version = client.get_latest_versions(
        f"EMI_Regression_{best_model_name}", stages=["None"]
    )[0].version

    client.transition_model_version_stage(
        name=f"EMI_Regression_{best_model_name}",
        version=latest_version,
        stage="Production"
    )

    print(f"\n✓ Model registered & moved to PRODUCTION → Version {latest_version}")

except Exception as e:
    print("\n⚠ Model registered but stage transition failed.")
    print(e)



✓ Model registered & moved to PRODUCTION → Version 1
